# Day 6 — Rate Limiting, Versioning & OpenAPI Docs

---

Three things every production API needs:

1. **Rate limiting** — protect against abuse, runaway clients, and cost overruns.
2. **Versioning** — change your API without breaking existing clients.
3. **Good docs** — let other developers actually use it.

FastAPI gives you the second and third for free (with a little tuning). For the first, we use `slowapi`.


In [ ]:
!pip install fastapi uvicorn httpx slowapi


## Why Rate Limit?

- **Cost** — every request costs CPU, bandwidth, maybe LLM tokens.
- **Abuse** — scrapers, brute-force login attempts, denial of service.
- **Fairness** — one bad client shouldn't degrade everyone else.

The response when a limit is exceeded: HTTP **429 Too Many Requests**.


## Rate Limiting with `slowapi`

`slowapi` is a port of Flask-Limiter for ASGI/FastAPI. You create a `Limiter`, register it on the app, and decorate routes.


In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.errors import RateLimitExceeded
from slowapi.util import get_remote_address

limiter = Limiter(key_func=get_remote_address)

app = FastAPI()
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

@app.get("/ping")
@limiter.limit("3/minute")
def ping(request: Request):     # NOTE: must accept `request: Request`
    return {"pong": True}

client = TestClient(app)
for i in range(5):
    r = client.get("/ping")
    print(f"req {i + 1} -> {r.status_code} {r.json() if r.status_code != 429 else r.text[:60]}")


### Key bits

- `key_func=get_remote_address` — limit per client IP.
- `@limiter.limit("3/minute")` — the magic decorator.
- The route function **must** accept `request: Request` (slowapi needs it to extract the key).
- After the limit is hit, you get **429**.


## Rate Limit Format

| String | Meaning |
|--------|---------|
| `"5/second"` | 5 requests per second |
| `"10/minute"` | 10 per minute |
| `"100/hour"` | 100 per hour |
| `"1000/day"` | 1000 per day |
| `"5/minute;100/hour"` | Combine: both must hold |


## API Versioning

Why version? Once clients depend on your API, you can't change response shapes without breaking them. **Versioning lets you evolve the API in parallel.**

The most common approach: **URL prefix** — `/v1/...` and `/v2/...`. FastAPI's `APIRouter` makes this trivial.


In [ ]:
from fastapi import APIRouter

app = FastAPI()

v1 = APIRouter(prefix="/v1", tags=["v1"])
v2 = APIRouter(prefix="/v2", tags=["v2"])

@v1.get("/items")
def items_v1():
    return [{"id": 1, "name": "pen"}]

@v2.get("/items")
def items_v2():
    # v2 adds price + stock
    return [{"id": 1, "name": "pen", "price": 2.5, "stock": 100}]

app.include_router(v1)
app.include_router(v2)

client = TestClient(app)
print("v1:", client.get("/v1/items").json())
print("v2:", client.get("/v2/items").json())


### Versioning Strategies

| Approach | Example | Trade-off |
|----------|---------|-----------|
| **URL prefix** | `/v1/users` | Easiest, very visible. |
| **Header** | `Accept: application/vnd.api+json; version=2` | Cleaner URLs, harder to debug. |
| **Query param** | `/users?v=2` | Discouraged — easy to forget. |

For bootcamp + most real APIs: **URL prefix**.


## OpenAPI & Swagger UI — Free Docs

Every FastAPI app exposes three URLs out of the box:

| URL | What it is |
|-----|------------|
| `/docs` | Interactive **Swagger UI** — click to try endpoints |
| `/redoc` | Cleaner reference-style docs |
| `/openapi.json` | The raw OpenAPI 3 schema (machine-readable) |

These are built from your route signatures and Pydantic models. You don't have to do anything to get them.


In [ ]:
app = FastAPI()

@app.get("/")
def root():
    return {"ok": True}

client = TestClient(app)
schema = client.get("/openapi.json").json()
print("OpenAPI version:", schema["openapi"])
print("Paths:", list(schema["paths"].keys()))


## Customizing Your Docs

Make your docs informative — title, version, description show up prominently in Swagger UI.


In [ ]:
app = FastAPI(
    title="Bookstore API",
    description="A tiny example showcasing OpenAPI customization.",
    version="1.0.0",
    contact={"name": "Team", "email": "team@example.com"},
)

@app.get("/")
def root():
    return {"ok": True}

client = TestClient(app)
print(client.get("/openapi.json").json()["info"])


## Tags — Grouping Endpoints in Swagger

`tags=["users"]` on a route puts it under a "users" heading in the docs.


In [ ]:
app = FastAPI(title="Tagged API")

@app.get("/users", tags=["users"])
def list_users():
    return []

@app.post("/users", tags=["users"])
def create_user():
    return {}

@app.get("/items", tags=["items"])
def list_items():
    return []

client = TestClient(app)
paths = client.get("/openapi.json").json()["paths"]
for path, methods in paths.items():
    for method, op in methods.items():
        print(f"{method.upper():6} {path:10} tags={op.get('tags')}")


## Per-Endpoint Summary & Description

Each route can have its own `summary` (one-liner) and `description` (longer text). The description supports Markdown.


In [ ]:
app = FastAPI()

@app.get(
    "/users/{user_id}",
    tags=["users"],
    summary="Fetch a single user",
    description="Returns a user by their **integer ID**. Returns 404 if not found.",
)
def get_user(user_id: int):
    return {"id": user_id}

client = TestClient(app)
op = client.get("/openapi.json").json()["paths"]["/users/{user_id}"]["get"]
print("summary:    ", op["summary"])
print("description:", op["description"])


## Deprecation

When an old endpoint should be retired, mark it `deprecated=True`. Swagger UI shows a strikethrough; clients can detect it in `/openapi.json`.


In [ ]:
app = FastAPI()

@app.get("/old", deprecated=True, summary="Use /new instead")
def old():
    return {"warning": "deprecated"}

@app.get("/new")
def new():
    return {"ok": True}

client = TestClient(app)
print("old deprecated?", client.get("/openapi.json").json()["paths"]["/old"]["get"]["deprecated"])


## Putting It Together

A single app with: customized metadata, v1+v2 routers, rate limiting on `/ping`, tags, and a deprecated endpoint.


In [ ]:
from fastapi import FastAPI, APIRouter, Request
from fastapi.testclient import TestClient
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.errors import RateLimitExceeded
from slowapi.util import get_remote_address

limiter = Limiter(key_func=get_remote_address)

app = FastAPI(
    title="Day 6 Demo",
    description="Rate limiting + versioning + custom docs.",
    version="1.0.0",
)
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

v1 = APIRouter(prefix="/v1", tags=["v1"])
v2 = APIRouter(prefix="/v2", tags=["v2"])

@v1.get("/items", summary="List items (v1)")
def items_v1():
    return [{"id": 1, "name": "pen"}]

@v2.get("/items", summary="List items (v2, enriched)")
def items_v2():
    return [{"id": 1, "name": "pen", "price": 2.5, "stock": 100}]

app.include_router(v1)
app.include_router(v2)

@app.get("/ping", tags=["util"])
@limiter.limit("3/minute")
def ping(request: Request):
    return {"pong": True}

@app.get("/old", deprecated=True, summary="Use /v2/items")
def old():
    return {"warning": "deprecated"}

client = TestClient(app)
print("info:", client.get("/openapi.json").json()["info"])
print("v1:", client.get("/v1/items").json())
print("v2:", client.get("/v2/items").json())
for i in range(4):
    print(f"ping {i + 1}: {client.get('/ping').status_code}")


## Quick Recap

- **Rate limiting** with `slowapi`: create a `Limiter`, decorate routes, accept `request: Request`. 429 on overflow.
- **Versioning** with `APIRouter(prefix="/v1")` — easy parallel evolution.
- **OpenAPI** is free: `/docs`, `/redoc`, `/openapi.json` work out of the box.
- Customize with `FastAPI(title=..., version=..., description=...)`, plus per-route `tags`, `summary`, `description`, `deprecated`.
- This concludes the backend section. You've now built APIs with validation, auth, middleware, rate limiting, versioning, and docs — the whole production toolkit.
